# MiniMax-H3 导演台全能工作流 · Colab Pro 运行版 **v4**

这个 notebook 只有一个目标：把 `MiniMax+H3+导演台全能工作流.json` 在 **Google Colab Pro** 上直接跑起来。
JSON 已内嵌，不用上传；节点、权重、文件名、素材、访问地址全部自动处理。

## v3.1 跑不了这个 JSON 的原因，以及 v4 的修法

| # | v3.1 的问题 | v4 的做法 |
| --- | --- | --- |
| 1 | 工作流核心节点 `MiniMaxH3Director` **根本没装**（只装了 KJNodes / VHS / 双时钟），打开只能看到红框 Missing Node Type | Cell 4 从 JSON 里反推节点类型，自动装 `AIMixer/ComfyUI_MiniMaxH3_Director` |
| 2 | 下的模型和 JSON 里写的对不上：JSON 要 `fl2va_pruned_int8_convrot` + `nvfp4` 文本编码器，v3.1 下的是非剪枝 34GB + int8 | Cell 5 先列举 HF 仓库真实文件，按 GPU 架构 + 剩余磁盘选，Cell 6 把实际文件名写回 JSON；fl2va 与 ref2va 两套底模一并下好 |
| 3 | `nvfp4` 文本编码器在 A100 上没有内核，直接报错 | 按 `sm` 自动降级：sm\_120+ 用 nvfp4，sm\_89+ 用 fp8，其余（含 A100 sm\_80）用 int8\_convrot |
| 4 | JSON 要自己拖进界面导入 | 已内嵌，自动写到 `user/default/workflows/`，侧栏直接点开 |
| 5 | 时间线引用的 4 张首尾帧是作者本地的哈希文件名，Colab 上不存在 | Cell 7 按时间线顺序上传，自动落到 `input/` 并回写文件名与真实尺寸 |
| 6 | 通道写死在启动流程里，连不上也不说为什么 | 默认走你自己的 frps（TCP 直连，不绕公共边缘节点）；frps 拒绝的原因直接翻译成人话，连不上自动兔子底到 Colab 端口代理 |
| 7 | 写死 A100 40G，Colab Pro 分到 L4 / T4 就崩 | Cell 1 识别档位，Cell 8 自动切 `--lowvram` / `--novram`，小显存还会打开分段清显存 |
| 8 | 装完不知道能不能跑，靠肉眼对下拉框 | Cell 8 启动后调 `/object_info` 逐个校验节点、模型文件名、素材，缺什么直接列出来 |

## 执行顺序

**Cell 1 → 8 依次跑一遍。** 之后只是重启界面的话，只需 **Cell 1 + Cell 8**。

| Cell | 做什么 | 耗时 |
| --- | --- | --- |
| 1 | 配置 + 环境自检（GPU / 内存 / 磁盘） | 秒 |
| 2 | 释放内嵌 JSON + 反推需要的节点 / 模型 / 素材 | 秒 |
| 3 | ComfyUI master + torch 探测 | 2–4 分钟 |
| 4 | 导演台插件等自定义节点 | 1–2 分钟 |
| 5 | 下模型（fl2va + ref2va 两套底模，约 76 GB，断点续传） | 15–35 分钟 |
| 6 | 改写工作流并装进 ComfyUI | 秒 |
| 7 | 上传首尾帧素材 | 看图片大小 |
| 8 | 启动 + 开通道 + 体检 | 1–3 分钟 |

## Colab Pro 运行时选哪个

菜单 → **修改 → 笔记本设置**：

| 运行时 | 显存 | 能不能跑 | 自动策略 |
| --- | --- | --- | --- |
| **A100 + 高 RAM** | 40 GB | 推荐 | 普通显存模式 + `--cache-none` |
| **L4** | 22.5 GB | 可以，慢 3 倍 | `--lowvram` + 分段清显存，建议降到 0.3 MP |
| **T4** | 16 GB | 不推荐 | `--novram`，大概率 OOM 或慢到不可用 |
| CPU | — | 不行 | Cell 1 直接抦下 |

磁盘：两套底模 42 GB + 文本编码器 27 GB + VAE，共 ≈ 76 GB，A100 运行时给的盘够用；只想跑 fl2v 就把 Cell 1 的 `download_both` 改成 `False`，省 21 GB。

另外：**一定要选高 RAM**，21 GB 底模 + 27 GB 文本编码器换出时会吃掉 50 GB 以上系统内存。


In [1]:
# ==========================================================
# Cell 1: 配置 + Colab 环境自检
#   每次连接运行时都必须先跑这一格（重启界面也只要 Cell 1 + Cell 8）
# ==========================================================
import base64, gzip, io, json, os, re, shutil, subprocess, sys, time
from importlib.util import find_spec

ROOT = "/content"                       # Colab 固定盘符；本地调试可改
IN_COLAB = find_spec("google.colab") is not None

CFG = {
    "root": ROOT,
    "comfy_dir": ROOT + "/ComfyUI",
    "hf_home": ROOT + "/hf_cache",       # 和 models/ 同盘，硬链接才不会占两份
    "workflow_json": ROOT + "/workflow.json",
    "workflow_name": "MiniMaxH3_导演台全能工作流",

    # --- 权重：留空 = Cell 5 按 GPU 架构 + 剩余磁盘自动选 ---
    "repo": "Comfy-Org/MiniMax-H3",
    "dit_file": "",
    "te_file": "",
    "allow_full_dit": False,   # True = 允许下 34GB 非剪枝 DiT（只有挂 Turbo LoRA 才需要）
    #   fl2v/t2v/i2v 走 fl2va 底模，r2v/v2v/rv2v 走 ref2va 底模，是两个文件。
    #   默认两套都下（多 ≈21GB），以后在导演台里换任务不用重新下模型。
    "download_both": True,

    # --- Turbo 4 步加速 LoRA（需要非剪枝 DiT，会自动插一个 LoraLoaderModelOnly 节点）---
    "use_turbo_lora": False,
    "lora_repo": "larryvrh/MiniMax-H3-Turbo-Lora",
    "lora_src": "minimax_h3_turbo_4step.safetensors",
    "lora_out": "minimax_h3_turbo_4step_comfyui.safetensors",

    # --- 外网访问方式 ---
    #   "frp"        : 连你自己的 frps，TCP 直连不绕边缘节点，延迟最低（默认）
    #   "colab"      : Colab 自带端口代理开新窗口，零配置
    #   "cloudflared": 临时 https 域名，要绕 CF 边缘，慢
    #   "none"       : 只在本机监听
    "tunnel": "frp",
    "local_port": 8188,
    "frp_host": "usoren.usdream.dpdns.org",
    "frp_port": 7000,
    "frp_token": "",           # frps.toml 里配了 auth.token 就填上，否则留空
    "remote_port": 8091,       # 必须在 frps 的 allowPorts 范围内
    "frp_ver": "0.56.0",
    "tunnel_fallback": True,   # frp 没连上就自动改用 Colab 端口代理，不至于白启动

    # --- Google Drive：模型/成片落盘，跨会话复用（Drive 读盘慢，按需开）---
    "use_drive": False,
    "drive_dir": "/content/drive/MyDrive/minimax_h3",
    "save_outputs_to_drive": False,

    # --- 加速模块（对应导演台 2026-08-07 新增的「加速版」示例工作流）---
    #   UNETLoader -> PathchSageAttentionKJ
    #              -> MiniMaxH3MemoryEfficientSageAttentionPatch -> 导演台
    #   两个节点都来自 KJNodes；装不上 sageattention 时会以旁路状态插入，不影响出片。
    "accel_sage": True,
    "accel_steps": 0,          # >0 就把导演台 steps 改成这个值（加速版示例是 20）

    # --- 其他 ---
    "sage_mode": "auto",       # auto = 自动装 sageattention；skip = 不装、也不插加速节点
    "install_manager": True,
    "vram_policy": "auto",     # auto / highvram / lowvram / novram
    "extra_args": "",          # 额外启动参数，一般留空
}
CFG["frp_dir"] = ROOT + "/frp_%s_linux_amd64" % CFG["frp_ver"]
CFG["cfg_path"] = ROOT + "/h3_cfg.json"

os.makedirs(ROOT, exist_ok=True)
os.makedirs(CFG["hf_home"], exist_ok=True)
os.environ["HF_HOME"] = CFG["hf_home"]
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # Xet 后端已够快，两者同开会打架


def save_cfg():
    with open(CFG["cfg_path"], "w") as f:
        json.dump(CFG, f, indent=2, ensure_ascii=False)


DERIVED = ("sm", "cap", "vram_gb", "gpu_name", "tier", "can_fp8", "can_nvfp4",
           "dit_file", "dit_files", "te_file", "vae_files", "lora_ready", "sage_ok",
           "wf_types", "assets", "task_type", "model_family", "workflow_installed")


def load_cfg():
    """只重跑 Cell 1 + Cell 8 时，把前面几格探测/下载的结果捞回来。"""
    if os.path.exists(CFG["cfg_path"]):
        old = json.load(open(CFG["cfg_path"]))
        for k in DERIVED:
            if old.get(k) not in (None, "", [], {}) and not CFG.get(k):
                CFG[k] = old[k]
    return CFG


def log(msg, tag="*"):
    print("[%s] %s" % (tag, msg), flush=True)


def sh(cmd, cwd=None, check=True, quiet=False):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout and not quiet:
        print(r.stdout.strip()[-4000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError("命令失败(%d): %s" % (r.returncode, cmd))
    return r.stdout or ""


def pip(pkgs):
    sh("%s -m pip install -q %s" % (sys.executable, pkgs), quiet=True)


def free_gb(path=None):
    return shutil.disk_usage(path or ROOT).free / 1024 ** 3


def ram_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return 0.0


# ---------- 工作流读写小工具（Cell 2 / 6 / 7 / 8 全都用它们，所以放在最前面）----------
LOADER_FIELDS = {          # 节点类型 -> (widget 下标, models 子目录)
    "UNETLoader": (0, "diffusion_models"),
    "CLIPLoader": (0, "text_encoders"),
    "VAELoader": (0, "vae"),
    "LoraLoaderModelOnly": (0, "loras"),
    "CheckpointLoaderSimple": (0, "checkpoints"),
}


def wf_load(path=None):
    return json.load(open(path or CFG["workflow_json"], encoding="utf-8"))


def wf_save(wf, path=None):
    with open(path or CFG["workflow_json"], "w", encoding="utf-8") as f:
        json.dump(wf, f, ensure_ascii=False, indent=1)


def wf_find(wf, ntype):
    return [n for n in wf["nodes"] if n.get("type") == ntype]


def wf_types(wf):
    return sorted({n.get("type") for n in wf["nodes"] if n.get("type")})


def wf_loaders(wf):
    """[(节点, widget下标, models子目录, 当前文件名), ...]"""
    out = []
    for n in wf["nodes"]:
        spec = LOADER_FIELDS.get(n.get("type"))
        if not spec:
            continue
        idx, sub = spec
        wv = n.get("widgets_values") or []
        if len(wv) > idx and isinstance(wv[idx], str):
            out.append((n, idx, sub, wv[idx]))
    return out


def director(wf):
    ns = wf_find(wf, "MiniMaxH3Director")
    return ns[0] if ns else None


def timeline_idx(node):
    """timeline_data 是唯一一个能解析成含 segments 的 JSON 字符串的 widget。"""
    for i, v in enumerate(node.get("widgets_values") or []):
        if isinstance(v, str) and v.strip().startswith("{") and "segments" in v:
            try:
                if isinstance(json.loads(v), dict):
                    return i
            except Exception:
                pass
    return -1


def timeline_get(node):
    i = timeline_idx(node)
    return (json.loads(node["widgets_values"][i]), i) if i >= 0 else (None, -1)


def timeline_set(node, tl, i):
    node["widgets_values"][i] = json.dumps(tl, ensure_ascii=False)


def widget_after(node, label, kind):
    """按「分组标题 + 类型」定位 widget，不写死下标（节点版本变了也不会错位）。"""
    wv = node.get("widgets_values") or []
    if label in wv:
        i = wv.index(label) + 1
        if i < len(wv) and isinstance(wv[i], kind) and not (isinstance(wv[i], bool) ^ (kind is bool)):
            return i
    return -1


def timeline_assets(tl):
    """时间线引用到的 input 素材文件名，按出现顺序去重。"""
    seen = []

    def add(v):
        if isinstance(v, str) and v and v not in seen:
            seen.append(v)

    for seg in tl.get("segments", []):
        add((seg.get("genImage") or {}).get("imageFile"))
        add((seg.get("endImage") or {}).get("imageFile"))
        for r in seg.get("refs") or []:
            add(r.get("imageFile") if isinstance(r, dict) else None)
    g = tl.get("global") or {}
    add((g.get("genImage") or {}).get("imageFile"))
    add((g.get("referenceVideo") or {}).get("videoFile"))
    for r in g.get("refs") or []:
        add(r.get("imageFile") if isinstance(r, dict) else None)
    for a in g.get("refAudios") or []:
        add(a.get("audioFile") if isinstance(a, dict) else None)
    add((tl.get("video") or {}).get("videoFile"))
    return seen


# ---------- GPU：只问 nvidia-smi，不 import torch ----------
#   在内核里 import torch 会白占 1~2GB 显存，还可能触发
#   "Only a single TORCH_LIBRARY can be used to register the namespace triton"
gpu_name, vram_gb, cap = "", 0, None
q = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
       check=False, quiet=True).strip()
if q and "," in q:
    parts = [p.strip() for p in q.splitlines()[0].split(",")]
    gpu_name = parts[0]
    m = re.search(r"(\d+)", parts[1])
    vram_gb = round(int(m.group(1)) / 1024) if m else 0
    if len(parts) > 2 and re.match(r"^\d+\.\d+$", parts[2]):
        cap = [int(x) for x in parts[2].split(".")]

CFG["gpu_name"], CFG["vram_gb"] = gpu_name, vram_gb
if cap:
    CFG["cap"], CFG["sm"] = cap, cap[0] * 10 + cap[1]

# ---------- 运行时档位判定 ----------
if not gpu_name:
    CFG["tier"] = "cpu"
elif vram_gb >= 38:
    CFG["tier"] = "a100"      # A100 40G / H100，最舒服
elif vram_gb >= 20:
    CFG["tier"] = "l4"        # L4 22.5G，要 --lowvram
else:
    CFG["tier"] = "t4"        # T4 16G，能不能跑要看运气

BAR = "=" * 64
print(BAR)
print("Colab 环境自检")
print(BAR)
log("运行环境   : %s" % ("Google Colab" if IN_COLAB else "非 Colab（本地/其他）"))
log("Python     : %s" % sys.version.split()[0])
log("GPU        : %s | 显存 %d GB%s"
    % (gpu_name or "未检测到", vram_gb,
       " | sm_%d" % CFG["sm"] if CFG.get("sm") else ""))
log("系统内存   : %.0f GB" % ram_gb())
log("可用磁盘   : %.0f GB" % free_gb())

if CFG["tier"] == "cpu":
    raise RuntimeError(
        "没有 GPU。菜单 → 修改 → 笔记本设置 → 硬件加速器选 GPU（Colab Pro 建议 A100 + 高 RAM）")
if CFG["tier"] == "a100":
    log("A100 档：21GB DiT + 27GB 文本编码器靠自动换进换出，正常模式即可")
elif CFG["tier"] == "l4":
    log("L4 档（22.5GB）：会自动加 --lowvram，速度约为 A100 的 1/3，建议把分辨率降到 0.3MP", "!")
else:
    log("T4 档（16GB）：无 bf16 硬件支持，H3 极易 OOM 或慢到不可用。"
        "Colab Pro 请在「修改 → 笔记本设置」里换 A100 / L4", "!")
if ram_gb() < 40:
    log("系统内存 < 40GB：模型换出时会挤爆内存，请把运行时改成「高 RAM」", "!")

# ---------- 可选：挂 Drive ----------
if CFG["use_drive"] and IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.makedirs(CFG["drive_dir"], exist_ok=True)
    log("Drive 已挂载：" + CFG["drive_dir"])

# ---------- 可选：HF Token（左侧 🔑 Secrets）----------
if IN_COLAB:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            log("已读取 HF_TOKEN")
    except Exception:
        log("未配置 HF_TOKEN（公开仓库不影响下载）")

save_cfg()
print(BAR)
log("配置已写入 " + CFG["cfg_path"])


Colab 环境自检
[*] 运行环境   : Google Colab
[*] Python     : 3.12.13
[*] GPU        : NVIDIA A100-SXM4-80GB | 显存 80 GB | sm_80
[*] 系统内存   : 167 GB
[*] 可用磁盘   : 189 GB
[*] A100 档：21GB DiT + 27GB 文本编码器靠自动换进换出，正常模式即可
[*] 已读取 HF_TOKEN
[*] 配置已写入 /content/h3_cfg.json


In [2]:
# ==========================================================
# Cell 2: 内嵌工作流 JSON + 反推运行需求
#   JSON 已经打包进本 notebook，不用手动上传。
#   想换自己的工作流：把 USE_EMBEDDED 改成 False，运行本格后上传。
# ==========================================================
USE_EMBEDDED = True

WF_GZ_B64 = (
    "H4sIAI3rdWoC/+1ceW8bR7L/KgQXeMguKHruQ8BiodhyIsCS8mTHD4vIGAw5PdTE1AwzHFLyOgbkZGMljuLj2UnWWTvO4Ssb"
    "JNZuDnt9Au+jvNVQ1F/5Cq+6e46e4YiUnEDIgxfQwanqrqqu/lV3sVnNk+Wm2Q6MpuMeNxyrPC5wlbLrWahdHn/tZHkRXpXH"
    "geR1glYnoETXXARieXr2wOShcqWMu2IGLx2DB7OGmgwzONHKtPXqZtP5E7KMjJBT0LMNZJAiKlxFE+C55WGhYyoYpHHwvORY"
    "DRS0ja7Z7BDryouO6yyay8aCaNhNoWsaLb/jgmjHDTSj7rld3wuqbdNGAXLbnt8G/RayzU4zKIM8x00GREWXx5OhgZiAWniq"
    "MkhLR8nSopHun51+cbZopKzMApVLyGksBIZFxKRaM+RUcY48SndWOAzebpoNGPtJUIQnnU9EvDozeeSQZ1rIx2KdoEkmCRw9"
    "bS6XXhZLmA+clu+1kB84eCIoSpoZbDzL1Dg+qgeefwK6W45td9qO5xqR6FMJAIwOwgJcaGvWsHUns6MbD/wOqjDeJgQY5wxI"
    "KmFKyfb80uH/mMuOFlp4Pn4xzuH5GQb8/YemXmFxrzC4j3jxhEQt8/NB6DnQ85ySRb0gFMP+jSXkit2mIQo1g/Gz27VbkmEu"
    "vZHza9Rk5+CvN51WHvwpLR0qSxsFQFZmgcoc6HNg3yHIIyEF4i3UdepEQXvBxELUSo6VKksIo9TFQgejScjM/0A0YWLphWha"
    "SvtK/4nnc+zood/uIK52OfdsTAVoOTCQWweZ/oiASmcriiY2sqJxDwsrZthpWPGjwuroxCQbVTITVZQVOTVql58PTM6FlKxl"
    "I0ocvZF0HQt5wEKG3eKVjDtHhA3uk4uahJSOhCGNAlgqcBBiIuuNAYQdxYMoUT/tYqUeMvYslKDJcAAlpg9DSWp5ChJhlyAR"
    "f3mQyNxIkJgdy4kdJQq/WpBIQ0EygQexe5AMGftegUQcBZLDR+amZl5iHJoQIn+kDfL+jDgMRhQJtmKCCQoSnVOqoshzsqRo"
    "qlzR+aoqchqvq5zAS5okFYEni4pIVdvr+HTrATTTxDu2N2FF9v6uyNSoUcHEa0nHV3xYrdHShHuCmfkD0SyV+k8v9D9f6529"
    "GV48O4CC7eZtm4liNKUzpY0M56kDk7NMQOtsPEe8GMNx04GYJgxmxgQNsikhO2Mar+q6yMNkKoJcESWxqigSL0ucLKhi8aQJ"
    "UkUrnDgIhQZqJxPHq6nNCSsyemp64qXChShqiP0zkJOQEEvFa6n4mBNJn3j1wFShS2i7wlzIbrWZxQc/JYr0VBGlR2oOHpqd"
    "OFKkhsoqUFJzIB1HrWChMOdKualClhb7bqZQKSN7EPlKumL6yAwQ2Ql3DG0yoCjNSdUMzXQYLSnsVeyUFO1gZt1resAo/0YQ"
    "hHLR7vJmaRE1zJazDCtv6c3SRLsFZsGLWSIF0kVI3J1WE/1eFH5benPefXNsbCz5hccSVxWgOa+M6/BP4bTSckmUhVLEE1Oe"
    "KirAk3gl5kkpT1MkzNO4mCenPF3hgCdLUsxTUh6shlgoVhsxVYbJgx3AlBKpGsMUeNJTTUzVWaaGdWKLYybbFeIWc5VIKV/l"
    "WKaqZJmMe3iZwz01UYiZzDh5hThBF5KerE5NwEyeEyI3CBmlusARrkb6ZpKBLFbjYxHTP255S+6MFyBmgcaP46XDgPrSYRQE"
    "jttol+aQjXxI33G7ZEWtNRJgcRxXTvMaDvIYnjlFUQSxKkqCpssyp2o8LIE8JwJJEERNVXRZBNKx6CxA3PFOkCJeGrXQ09yy"
    "4zfZNIehjdyY08ZYFUvMS9zBlkHbsVsGD1uGlm4ZAEuAKS+IqqoLOtnoRUmrihrP6zKnaKKsF+4ZVPK+6MDkZdGI91ojELpg"
    "idkJvPhf4d6SjIgsyfozD61w6XeaCL80Wj6ynWV2G8hxmI1ggDNypgbVFNni+YtmwJpACYzmmDAqBU5EFZ1lACTr7DkGeWbO"
    "MKLnVMfBPxoH/jgzMT21nyg0joqF5xlU7uAGpKYOMru73X6o1GgDyrsxJtPRDtuUUs1pgOrbBmi0PSuVwvSGBIN67BfJcJTC"
    "DIdo0I79nCRnIJkhMtlccnf5TPzoY/fXvY6bwWWGOjxXYdsWOoLm8Ebq8swbAGPHzs62Z8bgo5bnB4xbSF4cq0m4I4M6asks"
    "mTzsOJBms2f1gipXdVVRkrS6wvNKlRcEXoTNWlVkhewyg6smPiMu/e/K5dLWrY/C9Sfh/dubl6/3b5/Z+uLiCwcdvx2MHTLb"
    "Qekgdic+JoOfrdXV3mf3+t8+2Xz8bbnCVxRFwTGzjKxyBTJ3WVUqkDeQX1EVKuWT8+Uu8vGZ8nx5XK7Ml5HlBNMQEPA4X26j"
    "xiJyg/kyMAIvMJtEVRt4uPM8ncg5SPWAAtLn6TILDyAWB+oMsIkgIoEwDwI5JbU7NdtrQiimJOxz8kR2AUKyY62QLkRP02Yr"
    "fqaTTCzbjyFFbNGAAW/X4U24dZjw50y3Eck4Fduyv+m0ErGNplczm9T2wGwfPxLbsatZIPbCurbYCtIhwUqVqPHjhOVo6qoC"
    "xwx6r8Avee/hgcHaCZlRx+u0k9QIuLbZbCM8SORO4WCgeklcMHpPJd78L8cKFoAKiWJCe5l8ngBEnObRkZDDktSpdP2kshdj"
    "DBHwEXNNkr0DXByPcMRxqfTC5tcfbb73bu+z1XD1DPVemvFDK8jDMSlK8zHwMO6antuYtMgwMJLncewQezG+58sLsaWUuWgu"
    "Ty7jOE3Qy2GgE1ICdbPZpEbiISVU8BfyCb5T1zrBiUkX700W49iUNwvx1DRbiS4de8bvuIcBjfVgsGfCojFIIU3jjngWz5NF"
    "jFlsW8Gy+CdHEY4HXQqAwPSDaDywLTaID3gyOzYbD5RkdXzse1BXj4KdAerG/ZWN+3/bXPs+/OZCePVO79PTWyvXw5uX+/cu"
    "/vRoDf5ufvLn/tPVzTvvb9z/pv/lDWjWO/cetAyvrm/cPwttNh6f2Xj40cbDDzb//OPmw9ubD7+h3f+18lbScePR91ufnO+v"
    "/g23AYH3Lva+uRl+uU5G48LEB04XvZJaVTOtUrSmQAOnfRgPmLh2Ptr5MXXStXK0YTi3JNVW62pdkMSaKimmJKqWJYpmXeCR"
    "xSsivF2BlVoUdLMObwokZKlIqVmaWJMkjlNks9pyG8ScpUyQLLDhgeccuda2Npi8LdX0uq5JOm8pulLj4EcQaoKsaTVbE2q6"
    "opiKLsoKsuqcbSFdFupIFGVN4GxBqQ/YAG9rWBsUTcAmZNax7FIEm2EWWJrqvm6LYiMDLAqcnwmtBFTh069ZUIXn1/qrb1Ew"
    "9C6eCc/eoUgL312naAz/8dXm9e96H58Geu+dL+AHMBNeu7Bx/32gAJA2r630Pv5x6+Pvgd77ARB7Dbf8x5Vw/S+0fe/Hd8Mb"
    "f8fER+fD1Qf9S9fCG1/1317rXbk7Epb/WjlNdYGKcOXKxgO87mMDvlzfuv7dXiJ2j9AyFLCmxZl2vc7rHESLbGl1JNlcXUK8"
    "ZCsyz9dAad2sCaJlId4yOZHnNUG1JBNJvMpZ3E6DZheIXVo+0WnXze7rGcTSrf8XQezGw3MUgbC+9X+413/wNcYneRFeOLf5"
    "1VNM//zL3tUPMP3tNXjc+OetcO0djO1zD3sfXgm/vYHb3H2SALj/wzrAm7K2VtdwxywIQSzIBOxhgUQafv3Dev/eXYxMAt29"
    "ht8ezf0w/GmiWpNsyDoFTeRq9Roya3Ve1GyzXpfFOofglc6JnCZLoqVrdQFMUZGqQyeTV+uW9EvgD+/MgbOI4D0DSjIEnB1m"
    "Y6soBQEh0+YyPrpKSOBxOtCo/uBgDqPYmuPoRJr7bpMIGG061Xu+x22XfyhCPuII5XnIPkhON7CzJjOFCmZqD9Z2ZqLITPys"
    "mXomX8W57sDKs20WUojqvfVVLvd5Fmf9O/PZfbQkCCiMlj3YilgIaMqvO16SHKg4XvbWW7nU69kD5rlIvArRn85nEfr3Igli"
    "5lPkuV8Z+o8N6OOxwvaCN/qs5P99/kGm5bk/1Rh+bvHv3fhZsPO8nC8MP0F4LjamEUB4Xt7pnzp2Cn9Y9vVfNh/cph+ZlSu4"
    "0tBHbYN82NAOUIsUjiy2mqhcATyLlXJv5Xb/7cflCt2gyrPNprlo4s/dCmsWSL1oWkYmpZ8vxpxRt4Bou3xhB64yTcXK+cIR"
    "yh1e75u2ZIQnJaypcDH38fOOhKctGeG4dj6Vq2SvS4y+GUJaFd+RMNvHjfxFiYTG3JZgaCOvTDAyC1TSTwoNujowarP0VHWe"
    "PvJT5byColJCy2j4LaNtEnymNmTpTCFhjh7Z8OKBl+ZmXy10eV5DURmJ3WCLSOCJmVfyNKq0gEooEN1GyGJkk0emDoA+Dq8y"
    "iEQUVdmQ+gP8uR5baZMS81UNEXFknQQrtuguG16Q2Ets5Jm5vRY9Dx9WLKVAAV3jGA0RIVWREIbrSAQVKPGRbeCid1LvkKrK"
    "kNlSigx5uNqc6KJox0UIBj0TZQOeJTMxnyUPV54TXaQ8Ovs1LDMwWe0ZOqM+Rx8Z93kF28e9aXVNt05DZLCQONdmYBVgODte"
    "BxiNReEK22W70BjKYUI3eh4Ru5G8IlVkQfKLlUU8Rl1CGbXop3KLlNYXkNXZVm3CZRQztJGqGelFyhccOzDiCsYC9QyfMSBD"
    "HbV2ZXVsa0RcXLadEUkFW8aIXOnaCCOGVOlHWGwh3x6GfMIfQH1E3THiIy2FV0GR6RtdWCqMGgqWEHKNuGKj0Kxh7dlMaFir"
    "2OzZ2UOTEzPFOdIwqwqGQStgjHyR3KD9hQ1Tw7dhj7a42ICCe9jp1dGkhDiuIN55GSmLrqhqlKQI8WsmAiJSoX0RD2cu0cts"
    "uhbfDc2s5fHtiUxOFZeupmlDbApZ/WJB7L4UN4jWqqxYAti4RbKkxPYOQUfUhGYW0UNmL87qSbaC2MIkXY4IUfowrBh3cB7T"
    "olxl+3siklh8TyQRF9591Ht0OTy/3jv/3xsPfwyvrofXVn569MlCELTa4/v2NZxgoVOr1r3FfRNT084y8vft9xbtE69OGYPV"
    "6fPuvBveeqv36dWNx083L9/xhe78fBf/8eEvPvq5utK/dTpcPxM++LB/5mrvgy96714AzwldM983wN0c/Ad/YrxdX/KtALgr"
    "rS/sfXhl8877jPlLS0vVmtN08C8ZBC2sf/Eof+SNTl1pTM69uA937713KXy0snXpSe/czVx3v+O6jtsgXnD3/cFxu06A9sMU"
    "/d6SkdJRX+dLve/uhGfWtr44g4tZ514O758Oz9/deHgzfOeD/vf/3Lx8HSwNbz4Ob3wS/vXJ5nur1Njw0q2fHp1+5tsd283c"
    "Nhc7FFksZy6siszVNnxjtcrzokDudCgyX9V0GV7rmoZJ8ZUO6RmudMh4fULLgW/iDqR8GALCgkWtdgI60kG7eGiZL4hILhf+"
    "z70SvfMASuLiY+jBCZJSPTo5d3hqdgYoXIWr4JsUtu+5AXKto7RGF2TyVUGs4jstFrHYs+02XtRf42VJr3KKymmarCmCLIuV"
    "MUnWqjqMGF/VEHReUHDFehsWYTCOr0q8IOgCvuYiSJLM4+OxcsP3Oi16ipF4WrRVTcczZYMt0XogSIVflhFNZe/O5+Gn74dn"
    "P+s/flx6IfUCrlGueR3XAvyRWYJpk+AHfmWFwwU+qVZtQhupUij8Lg5mZ2CVAT6wMp6H/3peW00erU1MtfWfXApXH2QVaHQw"
    "eEwqFn8sLS5/Ta8oMKEq/EY3VI5VXuPFilTBpeIVGAY+SsE0qcJTGrSkx0KYKldESuXTlkpFoDQwi5yaYKJKu2JlUVE8pmq0"
    "q4K702sDmKpT1VhOlJEBVSAi5YqGJURvVI5F4RJfdx5bEMfiG81jFMsQLbbToK7qxlDFpbv0W3TwF+cYccThC7FRg1P/B2Kv"
    "weJmRwAA"
)

WF_PATH = CFG["workflow_json"]

if USE_EMBEDDED:
    data = gzip.decompress(base64.b64decode("".join(WF_GZ_B64.split())))
    open(WF_PATH, "wb").write(data)
    log("已释放内嵌工作流 -> " + WF_PATH)
elif not os.path.exists(WF_PATH):
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    shutil.move(name, WF_PATH)
    log("已接收上传的工作流 -> " + WF_PATH)


# ---------- 工作流读写小工具（Cell 6 / 7 复用）----------
LOADER_FIELDS = {          # 节点类型 -> (widget 下标, models 子目录)
    "UNETLoader": (0, "diffusion_models"),
    "CLIPLoader": (0, "text_encoders"),
    "VAELoader": (0, "vae"),
    "LoraLoaderModelOnly": (0, "loras"),
    "CheckpointLoaderSimple": (0, "checkpoints"),
}


def wf_load(path=None):
    return json.load(open(path or WF_PATH, encoding="utf-8"))


def wf_save(wf, path=None):
    with open(path or WF_PATH, "w", encoding="utf-8") as f:
        json.dump(wf, f, ensure_ascii=False, indent=1)


def wf_find(wf, ntype):
    return [n for n in wf["nodes"] if n.get("type") == ntype]


def wf_types(wf):
    return sorted({n.get("type") for n in wf["nodes"] if n.get("type")})


def wf_loaders(wf):
    """[(node, widget下标, 子目录, 当前文件名), ...]"""
    out = []
    for n in wf["nodes"]:
        spec = LOADER_FIELDS.get(n.get("type"))
        if not spec:
            continue
        idx, sub = spec
        wv = n.get("widgets_values") or []
        if len(wv) > idx and isinstance(wv[idx], str):
            out.append((n, idx, sub, wv[idx]))
    return out


def director(wf):
    ns = wf_find(wf, "MiniMaxH3Director")
    return ns[0] if ns else None


def timeline_idx(node):
    """timeline_data 是唯一一个能解析成含 segments 的 JSON 字符串的 widget。"""
    for i, v in enumerate(node.get("widgets_values") or []):
        if isinstance(v, str) and v.strip().startswith("{") and "segments" in v:
            try:
                if isinstance(json.loads(v), dict):
                    return i
            except Exception:
                pass
    return -1


def timeline_get(node):
    i = timeline_idx(node)
    return (json.loads(node["widgets_values"][i]), i) if i >= 0 else (None, -1)


def timeline_set(node, tl, i):
    node["widgets_values"][i] = json.dumps(tl, ensure_ascii=False)


def widget_after(node, label, kind):
    """按「分组标题 + 类型」定位 widget，不写死下标（节点版本变了也不会错位）。"""
    wv = node.get("widgets_values") or []
    if label in wv:
        i = wv.index(label) + 1
        if i < len(wv) and isinstance(wv[i], kind) and not isinstance(wv[i], bool) ^ (kind is bool):
            return i
    return -1


def timeline_assets(tl):
    """时间线里引用到的 input 素材文件名，按出现顺序去重。"""
    seen = []

    def add(v):
        if isinstance(v, str) and v and v not in seen:
            seen.append(v)

    for seg in tl.get("segments", []):
        add((seg.get("genImage") or {}).get("imageFile"))
        add((seg.get("endImage") or {}).get("imageFile"))
        for r in seg.get("refs") or []:
            add(r.get("imageFile") if isinstance(r, dict) else None)
    g = tl.get("global") or {}
    add((g.get("genImage") or {}).get("imageFile"))
    add((g.get("referenceVideo") or {}).get("videoFile"))
    for r in g.get("refs") or []:
        add(r.get("imageFile") if isinstance(r, dict) else None)
    for a in g.get("refAudios") or []:
        add(a.get("audioFile") if isinstance(a, dict) else None)
    add((tl.get("video") or {}).get("videoFile"))
    return seen


# ---------- 解析本工作流 ----------
wf = wf_load()
CFG["wf_types"] = wf_types(wf)
d = director(wf)
tl, _i = timeline_get(d) if d else (None, -1)

print(BAR)
log("节点类型   : " + ", ".join(CFG["wf_types"]))
log("模型文件   :")
for _n, _i2, sub, fn in wf_loaders(wf):
    log("   models/%-16s %s   (%s)" % (sub + "/", fn, _n.get("title") or _n["type"]))

if d:
    task = (d["widgets_values"] or [""])[0]
    CFG["task_type"] = task
    fam = "ref2va" if re.match(r"^(r2v|v2v|rv2v)", str(task)) else "fl2va"
    CFG["model_family"] = fam
    log("导演台任务 : %s  -> 需要 %s 系底模" % (task, fam))

if tl:
    CFG["assets"] = timeline_assets(tl)
    o = tl.get("output", {})
    log("输出规格   : %sx%s | %s 帧 | %s fps | %d 段"
        % (o.get("width"), o.get("height"), tl.get("totalFrames"),
           tl.get("frameRate"), len(tl.get("segments", []))))
    log("需要素材   : %d 个（Cell 7 上传）" % len(CFG["assets"]))
    for a in CFG["assets"]:
        log("   " + a)
else:
    CFG["assets"] = []

save_cfg()
print(BAR)


[*] 已释放内嵌工作流 -> /content/workflow.json
[*] 节点类型   : CLIPLoader, CreateVideo, MarkdownNote, MiniMaxH3Director, PreviewAny, SaveVideo, UNETLoader, VAELoader
[*] 模型文件   :
[*]    models/diffusion_models/ minimax_h3_fl2va_pruned_int8_convrot.safetensors   (MiniMax H3 UNET)
[*]    models/text_encoders/   qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors   (CLIP (minimax / Qwen3-VL))
[*]    models/vae/             minimax_h3_video_vae_fp16.safetensors   (Video VAE)
[*]    models/vae/             minimax_h3_audio_vae_fp32.safetensors   (Audio VAE)
[*] 导演台任务 : fl2v — 首尾帧生视频(First-Last Frame)  -> 需要 fl2va 系底模
[*] 输出规格   : 576x736 | 372 帧 | 24 fps | 3 段
[*] 需要素材   : 4 个（Cell 7 上传）
[*]    d47f7c7c243b746a437dd33ac21ed163192540329ac0784ed7e6bd83b440065a.png
[*]    a1f4b9c98491d696b06b022b2588bf82b966a69356edc0fde952ce335820f26c.png
[*]    aad0afcc1907dd5d8ce4f0c4e14f6511bde9cab23dde1da0311827d4ae4170d0.png
[*]    837b4f3722830bcbeabc138facc53c0e8fa90308543d98c2dde7e797b4a17cd4.png


In [3]:
# ==========================================================
# Cell 3: ComfyUI 本体（master）+ torch 探测
#   MiniMax H3 的核心节点来自 PR #15224，必须用 master 最新提交
#   全程不在内核里 import torch，一律丢子进程探测
# ==========================================================
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as pkg_version

COMFY = CFG["comfy_dir"]

if not os.path.exists(COMFY):
    log("clone ComfyUI ...")
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI " + COMFY)
else:
    log("更新 ComfyUI ...")
    sh("git fetch --depth 1 origin master && git checkout -q master "
       "&& git reset --hard -q origin/master", cwd=COMFY)
log("当前提交: " + sh("git log -1 --format='%h %ad %s' --date=short",
                      cwd=COMFY, quiet=True).strip())


def torch_on_disk():
    try:
        return pkg_version("torch")
    except PackageNotFoundError:
        return None


_before = torch_on_disk()
log("安装 requirements（装完校验 torch 有没有被换掉）...")
pip("-r %s/requirements.txt huggingface_hub hf_xet" % COMFY)
_after = torch_on_disk()
if _before and _after and _before != _after:
    log("torch 被改动：%s -> %s；若后面报 kernel image 错误，"
        "执行 pip install -q torch==%s 并重启运行时" % (_before, _after, _before), "!")

# ---------- 子进程探测 ----------
PROBE = "\n".join([
    "import json, torch",
    "info = {'torch': torch.__version__, 'cuda': torch.version.cuda,",
    "        'avail': torch.cuda.is_available()}",
    "if info['avail']:",
    "    p = torch.cuda.get_device_properties(0)",
    "    info['name'] = torch.cuda.get_device_name(0)",
    "    info['cap'] = list(torch.cuda.get_device_capability(0))",
    "    info['vram'] = round(p.total_memory / 1024 ** 3)",
    "    info['archs'] = torch.cuda.get_arch_list()",
    "print('PROBE=' + json.dumps(info))",
])
open(ROOT + "/probe_gpu.py", "w").write(PROBE + "\n")
out = sh("%s %s/probe_gpu.py" % (sys.executable, ROOT), check=False, quiet=True)
probe = None
for line in out.splitlines():
    if line.startswith("PROBE="):
        probe = json.loads(line[6:])
if probe is None or not probe.get("avail"):
    log("子进程输出：\n" + out[-2000:], "!")
    raise RuntimeError("torch 看不到 GPU，检查运行时类型后重跑本格")

cap = tuple(probe["cap"])
CFG["cap"], CFG["sm"], CFG["vram_gb"] = list(cap), cap[0] * 10 + cap[1], probe["vram"]
CFG["can_nvfp4"] = CFG["sm"] >= 120      # nvfp4 只有 Blackwell 有原生内核
CFG["can_fp8"] = CFG["sm"] >= 89         # fp8 需要 Ada / Hopper 以上
log("torch %s | cuda %s" % (probe["torch"], probe["cuda"]))
log("GPU: %s | sm_%d | %d GB" % (probe["name"], CFG["sm"], probe["vram"]))
log("量化可用性 : nvfp4=%s  fp8=%s  int8_convrot=是"
    % ("是" if CFG["can_nvfp4"] else "否", "是" if CFG["can_fp8"] else "否"))
if not any("sm_%d" % CFG["sm"] in a for a in probe.get("archs", [])):
    log("torch 编译目标不含 sm_%d：%s，大概率会报 no kernel image"
        % (CFG["sm"], probe.get("archs")), "!")

hit = sh("grep -rl MiniMaxH3 %s/comfy_extras %s/nodes.py || true" % (COMFY, COMFY),
         check=False, quiet=True).strip()
log("ComfyUI 原生 MiniMax H3 支持已就绪" if hit
    else "没找到 MiniMax H3 原生节点，代码不够新，重跑本格", "*" if hit else "!")

save_cfg()
log("Cell 3 完成")


[*] clone ComfyUI ...
Cloning into '/content/ComfyUI'...
[*] 当前提交: dd79c64 2026-08-07 Minimum officially supported pytorch is now 2.7 (#15413)
[*] 安装 requirements（装完校验 torch 有没有被换掉）...
[*] torch 2.11.0+cu128 | cuda 12.8
[*] GPU: NVIDIA A100-SXM4-80GB | sm_80 | 79 GB
[*] 量化可用性 : nvfp4=否  fp8=否  int8_convrot=是
[*] ComfyUI 原生 MiniMax H3 支持已就绪
[*] Cell 3 完成


In [4]:
# ==========================================================
# Cell 4: 按工作流实际用到的节点类型装插件
#   本工作流的关键缺件是 MiniMaxH3Director（导演台），v3.1 完全没装，
#   所以打开 JSON 只会看到一个红框 "Missing Node Type"。
# ==========================================================
COMFY = CFG["comfy_dir"]

# 节点类型 -> 提供它的仓库
NODE_REPOS = {
    "MiniMaxH3Director": "https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git",
    "PathchSageAttentionKJ": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "MiniMaxH3MemoryEfficientSageAttentionPatch": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "VHS_LoadVideo": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "VHS_VideoCombine": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "MiniMaxH3DualClockSampler": "https://github.com/shuaixn/ComfyUI-MiniMaxH3DualClockSampler.git",
}

repos = []
for t in CFG.get("wf_types", []):
    r = NODE_REPOS.get(t)
    if r and r not in repos:
        repos.append(r)
        log("工作流需要 %s -> %s" % (t, r.rsplit("/", 1)[-1]))
#   加速模块的两个节点不在内嵌工作流里，wf_types 反推不到，这里显式补上 KJNodes
if CFG.get("accel_sage") and CFG["sage_mode"] != "skip":
    kj = NODE_REPOS["PathchSageAttentionKJ"]
    if kj not in repos:
        repos.append(kj)
        log("加速模块需要 KJNodes -> ComfyUI-KJNodes")
if CFG["use_turbo_lora"]:
    repos.append("https://github.com/shuaixn/ComfyUI-MiniMaxH3DualClockSampler.git")
if CFG["install_manager"]:
    repos.append("https://github.com/ltdrdata/ComfyUI-Manager.git")

cn_dir = os.path.join(COMFY, "custom_nodes")
os.makedirs(cn_dir, exist_ok=True)

for repo in dict.fromkeys(repos):
    name = repo.rstrip("/").split("/")[-1].replace(".git", "")
    path = os.path.join(cn_dir, name)
    if os.path.exists(path):
        log("更新 " + name)
        sh("git pull -q", cwd=path, check=False, quiet=True)
    else:
        log("安装 " + name)
        sh("git clone --depth 1 -q %s %s" % (repo, path))
    req = os.path.join(path, "requirements.txt")
    if os.path.exists(req):
        # 过滤掉会顺手升/降 torch 的行，保住 Colab 原版
        skip = ("torch", "torchvision", "torchaudio", "triton", "#")
        keep = [l for l in open(req)
                if l.strip() and not l.lower().lstrip().startswith(skip)]
        if keep:
            open("/tmp/req.txt", "w").writelines(keep)
            pip("-r /tmp/req.txt")

# ---------- Sage Attention（加速模块的依赖）----------
#   PathchSageAttentionKJ 把注意力后端换成 sageattn；
#   MiniMaxH3MemoryEfficientSageAttentionPatch 换掉 H3 自注意力实现以压低峰值显存。
#   后者在 KJNodes 里标 EXPERIMENTAL，支持 sm80/86/89/90/120（A100 sm_80、L4 sm_89 都在内）。
SAGE_PROBE = "import sageattention as s;print('SAGE_OK=' + getattr(s,'__version__','1.x'))"


def sage_version():
    out = sh('%s -c "%s"' % (sys.executable, SAGE_PROBE), check=False, quiet=True)
    for line in out.splitlines():
        if line.startswith("SAGE_OK="):
            return line[8:].strip()
    return None


if CFG["sage_mode"] == "skip":
    CFG["sage_ok"], CFG["sage_ver"] = False, ""
    log("sage_mode=skip：不装 sageattention，Cell 6 也不会插加速节点")
else:
    ver = sage_version()
    if not ver:
        log("安装 sageattention（纯 Triton 轮子，不会动 torch）...")
        pip("--no-deps sageattention")
        ver = sage_version() or ""
    if not ver:
        pip("--no-deps sageattention==1.0.6")
        ver = sage_version() or ""
    CFG["sage_ok"], CFG["sage_ver"] = bool(ver), ver
    if ver:
        log("sageattention %s 可用" % ver)
        if ver.split(".")[0] == "1":
            log("PyPI 上只有 1.x：PatchSageAttentionKJ 能用；"
                "MiniMaxH3MemoryEfficientSageAttentionPatch 要 2.x 内核，"
                "启动后若报错就在界面里选中它按 Ctrl+B 旁路", "!")
    else:
        log("sageattention 装不上，加速节点会以旁路状态插入，不影响正常出片", "!")
    if CFG.get("sm") and CFG["sm"] not in (80, 86, 89, 90, 120):
        log("sm_%d 不在加速节点的支持列表（80/86/89/90/120）里" % CFG["sm"], "!")

# ---------- ComfyUI-Manager 离线模式：避免启动时卡在联网拉节点列表 ----------
import configparser
for p in (COMFY + "/user/__manager/config.ini",
          COMFY + "/user/default/ComfyUI-Manager/config.ini",
          COMFY + "/custom_nodes/ComfyUI-Manager/config.ini"):
    os.makedirs(os.path.dirname(p), exist_ok=True)
    cp = configparser.ConfigParser()
    if os.path.exists(p):
        cp.read(p)
    cp.setdefault("default", {})
    cp["default"]["network_mode"] = "private"
    cp.write(open(p, "w"))

save_cfg()
log("Cell 4 完成（新节点要重启 ComfyUI 才生效，即重跑 Cell 8）")


[*] 工作流需要 MiniMaxH3Director -> ComfyUI_MiniMaxH3_Director.git
[*] 加速模块需要 KJNodes -> ComfyUI-KJNodes
[*] 安装 ComfyUI_MiniMaxH3_Director
[*] 安装 ComfyUI-KJNodes
[*] 安装 ComfyUI-Manager
[*] 安装 sageattention（纯 Triton 轮子，不会动 torch）...
[*] sageattention 1.x 可用
[!] PyPI 上只有 1.x：PatchSageAttentionKJ 能用；MiniMaxH3MemoryEfficientSageAttentionPatch 要 2.x 内核，启动后若报错就在界面里选中它按 Ctrl+B 旁路
[*] Cell 4 完成（新节点要重启 ComfyUI 才生效，即重跑 Cell 8）


In [5]:
# ==========================================================
# Cell 5: 模型下载（列举 HF 仓库真实文件 -> 按 GPU/磁盘自动选 -> 硬链到 models/）
#   v3.1 把文件名写死，工作流里填的又是另一个名字，启动后下拉框对不上。
#   现在下完会把实际文件名回写进 CFG，Cell 6 直接改 JSON。
# ==========================================================
import struct
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi, hf_hub_download

COMFY, REPO = CFG["comfy_dir"], CFG["repo"]
sm = CFG.get("sm", 80)
fam = CFG.get("model_family", "fl2va")

api = HfApi()
try:
    info = api.model_info(REPO, files_metadata=True)
    SIZES = {s.rfilename: (s.size or 0) for s in info.siblings}
    FILES = list(SIZES)
except Exception as e:
    log("拿不到文件大小（%r），改用文件列表" % e, "!")
    FILES, SIZES = api.list_repo_files(REPO), {}


def pick(patterns, pool=None):
    for pat in patterns:
        for f in (pool or FILES):
            if re.search(pat, f):
                return f
    return None


def gb(f):
    return SIZES.get(f, 0) / 1024 ** 3


# ---------- 1. DiT：fl2va 和 ref2va 是两套底模 ----------
#   t2v / i2v / fl2v      -> minimax_h3_fl2va_*
#   r2v / v2v / rv2v      -> minimax_h3_ref2va_*
#   导演台里换任务类型 = 换 UNETLoader 里的文件，两套都备好就不用重下 21GB。
#   pruned 把调制权重压成查表，质量不变、体积减半；
#   只有要挂 Turbo LoRA 时必须用非剪枝版（LoRA 补的是完整 adaln_proj）。
want_full = CFG["use_turbo_lora"] or CFG["allow_full_dit"]


def dit_prefs(f):
    if want_full:
        return [r"diffusion_models/minimax_h3_%s_int8_convrot" % f,
                r"diffusion_models/minimax_h3_%s_bf16" % f]
    pref = []
    if sm >= 120:
        pref.append(r"diffusion_models/minimax_h3_%s_pruned_nvfp4" % f)   # Blackwell 专享
    pref += [r"diffusion_models/minimax_h3_%s_pruned_int8_convrot" % f,
             r"diffusion_models/minimax_h3_%s_int8_convrot" % f]
    return pref

# ---------- 2. 文本编码器 ----------
if sm >= 120:
    te_pref = [r"text_encoders/.*nvfp4", r"text_encoders/.*fp8", r"text_encoders/.*int8_convrot"]
elif sm >= 89:
    te_pref = [r"text_encoders/.*fp8", r"text_encoders/.*int8_convrot"]
else:
    te_pref = [r"text_encoders/.*int8_convrot"]      # A100 sm_80 只能走这条

fams = [fam] + ([f for f in ("fl2va", "ref2va") if f != fam]
                if CFG.get("download_both", True) else [])
dits = {}
for f in fams:
    got = pick(dit_prefs(f))
    if got:
        dits[f] = got
    else:
        log("仓库里没找到 %s 系底模，跳过" % f, "!")
if CFG["dit_file"]:
    dits[fam] = CFG["dit_file"]          # 手动指定的优先
dit = dits.get(fam)
te = CFG["te_file"] or pick(te_pref)
vae_v = pick([r"vae/.*video.*fp16", r"vae/.*video"])
vae_a = pick([r"vae/.*audio.*fp32", r"vae/.*audio"])
if not all([dit, te, vae_v, vae_a]):
    raise RuntimeError("仓库里没找齐文件：dit=%s te=%s vae=%s/%s" % (dit, te, vae_v, vae_a))

TASKS = [(REPO, dit, "diffusion_models", None),
         (REPO, te, "text_encoders", None),
         (REPO, vae_v, "vae", None),
         (REPO, vae_a, "vae", None)]
if CFG["use_turbo_lora"]:
    TASKS.append((CFG["lora_repo"], CFG["lora_src"], "loras", CFG["lora_out"]))

need = sum(gb(f) for _r, f, _d, _n in TASKS if _r == REPO) or 55
print(BAR)
log("当前任务   : %s  -> %s 系" % (CFG.get("task_type", "?"), fam))
log("DiT 主     : %s  %.1f GB" % (os.path.basename(dit), gb(dit)))
log("文本编码器 : %s  %.1f GB" % (os.path.basename(te), gb(te)))
log("VAE        : %s + %s" % (os.path.basename(vae_v), os.path.basename(vae_a)))

#   另一套底模：换任务类型时要用，磁盘够就一并下了
for f, p in [(k, v) for k, v in dits.items() if k != fam]:
    if free_gb() < need + gb(p) + 10:
        log("磁盘只剩 %.0f GB，%s 系底模（%.0f GB）这次不下；"
            "腾出空间后重跑本格即可补上" % (free_gb(), f, gb(p)), "!")
        continue
    TASKS.append((REPO, p, "diffusion_models", None))
    need += gb(p)
    log("DiT 备     : %s  %.1f GB   （r2v/v2v/rv2v 用）"
        % (os.path.basename(p), gb(p)))

log("合计 ≈ %.0f GB，当前可用 %.0f GB（HF 缓存与 models/ 同盘，硬链不翻倍）"
    % (need, free_gb()))
if free_gb() < need + 6:
    raise RuntimeError(
        "磁盘不够。处理：Cell 1 把 download_both 设为 False 只下当前任务这一套，"
        "或把 use_turbo_lora / allow_full_dit 设为 False 用剪枝版，"
        "或断开重连一个干净的 Colab 运行时")
if CFG["use_turbo_lora"] and "pruned" in dit:
    raise RuntimeError("Turbo LoRA 必须配非剪枝 DiT，否则 51 处 adaln_proj 形状不匹配")


# ---------- 3. 落盘工具 ----------
def link_real(src, final):
    """HF 缓存里是相对符号链接，先 realpath 解成真 blob 再硬链，
    否则换目录后立刻悬空 -> getsize 报 [Errno 2]。"""
    real = os.path.realpath(src)
    if not os.path.isfile(real):
        raise RuntimeError("HF 缓存解析失败: %s -> %s" % (src, real))
    if os.path.lexists(final) and not os.path.exists(final):
        os.unlink(final)                       # 清掉上一轮的悬空链接
    if os.path.lexists(final):
        return real
    try:
        os.link(real, final)
    except OSError:
        try:
            os.symlink(real, final)
        except OSError:
            shutil.copy2(real, final)
    return real


def safetensors_ok(path):
    size = os.path.getsize(os.path.realpath(path))
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        if n <= 0 or n + 8 > size:
            return False, size
        json.loads(f.read(n))
    return True, size


def add_diffusion_prefix(src, dst, prefix="diffusion_model."):
    """作者的 LoRA 键是 blocks.0.*，ComfyUI 要 diffusion_model. 命名空间。
    只重写 JSON 头，数据区原样拷贝，不用 import torch。"""
    tmp = dst + ".part"
    with open(src, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        head = json.loads(f.read(n))
        meta = head.pop("__metadata__", None)
        new, renamed = {}, 0
        for k, v in head.items():
            if k.startswith(prefix):
                new[k] = v
            else:
                new[prefix + k] = v
                renamed += 1
        if meta is not None:
            new = dict([("__metadata__", meta)] + list(new.items()))
        hb = json.dumps(new, separators=(",", ":")).encode("utf-8")
        hb += b" " * ((-len(hb)) % 8)
        with open(tmp, "wb") as o:
            o.write(struct.pack("<Q", len(hb)))
            o.write(hb)
            shutil.copyfileobj(f, o, 8 * 1024 ** 2)
    os.replace(tmp, dst)
    log("LoRA 键名已加前缀：%d 个 tensor" % renamed)


def fetch(repo, filename, subdir, rename):
    dest_dir = os.path.join(COMFY, "models", subdir)
    os.makedirs(dest_dir, exist_ok=True)
    base = rename or os.path.basename(filename)
    final = os.path.join(dest_dir, base)
    if os.path.exists(final) and os.path.getsize(os.path.realpath(final)) > 100 * 1024 ** 2:
        return base, "跳过(已存在) %.1f GB" % (os.path.getsize(os.path.realpath(final)) / 1024 ** 3)
    src = hf_hub_download(repo_id=repo, filename=filename, repo_type="model")
    if rename and repo == CFG["lora_repo"]:
        add_diffusion_prefix(os.path.realpath(src), final)
    else:
        link_real(src, final)
    ok, size = safetensors_ok(final)
    if not ok:
        raise RuntimeError("%s 头部校验失败，文件不完整" % base)
    return base, "完成 %.1f GB" % (size / 1024 ** 3)


# ---------- 4. 开跑（并行 2 路 + 断点续传）----------
t0, failed = time.time(), []
with ThreadPoolExecutor(max_workers=2) as ex:
    futs = {ex.submit(fetch, r, f, d, rn): f for r, f, d, rn in TASKS}
    for fu in as_completed(futs):
        try:
            base, msg = fu.result()
            log("%-58s %s" % (base, msg))
        except Exception as e:
            failed.append(futs[fu])
            log("下载失败 %s -> %r" % (futs[fu], e), "!")

if failed:
    log("以下文件没拿到，重跑本格可断点续传：\n  " + "\n  ".join(failed), "!")
else:
    log("全部文件就绪")

# 把实际文件名交给 Cell 6 回写 JSON（只登记真落盘了的）
DIT_DIR = os.path.join(COMFY, "models", "diffusion_models")
CFG["dit_files"] = dict((f, os.path.basename(p)) for f, p in dits.items()
                        if os.path.exists(os.path.join(DIT_DIR, os.path.basename(p))))
for f in ("fl2va", "ref2va"):
    if f in CFG["dit_files"]:
        log("%-6s 就绪：%s" % (f, CFG["dit_files"][f]))
    else:
        log("%-6s 未下载（需要时把 download_both 开着重跑 Cell 5）" % f, "!")
CFG["dit_file"], CFG["te_file"] = dit, te
CFG["vae_files"] = {"video": os.path.basename(vae_v), "audio": os.path.basename(vae_a)}
CFG["lora_ready"] = CFG["use_turbo_lora"] and not failed
save_cfg()
sh("ls -lLh %s/models/diffusion_models %s/models/text_encoders %s/models/vae"
   % (COMFY, COMFY, COMFY), check=False)
log("耗时 %.1f 分钟，剩余磁盘 %.0f GB" % ((time.time() - t0) / 60, free_gb()))
print(BAR)


[*] 当前任务   : fl2v — 首尾帧生视频(First-Last Frame)  -> fl2va 系
[*] DiT 主     : minimax_h3_fl2va_pruned_int8_convrot.safetensors  19.5 GB
[*] 文本编码器 : qwen3vl_32b_minimax_h3_int8_convrot.safetensors  25.3 GB
[*] VAE        : minimax_h3_video_vae_fp16.safetensors + minimax_h3_audio_vae_fp32.safetensors
[*] DiT 备     : minimax_h3_ref2va_pruned_int8_convrot.safetensors  19.5 GB   （r2v/v2v/rv2v 用）
[*] 合计 ≈ 70 GB，当前可用 187 GB（HF 缓存与 models/ 同盘，硬链不翻倍）


diffusion_models/minimax_h3_fl2va_pruned(…): reconstructing file:   0%|          |  0.00B / 21.0GB            

text_encoders/qwen3vl_32b_minimax_h3_int(…): reconstructing file:   0%|          |  0.00B / 27.1GB            

text_encoders/qwen3vl_32b_minimax_h3_int(…): downloading bytes:           |  0.00B            

diffusion_models/minimax_h3_fl2va_pruned(…): downloading bytes:           |  0.00B            

...

vae/minimax_h3_audio_vae_fp32.safetensor(…): downloading bytes:           |  0.00B            

[*] minimax_h3_audio_vae_fp32.safetensors                      完成 0.6 GB


diffusion_models/minimax_h3_ref2va_prune(…): reconstructing file:   0%|          |  0.00B / 21.0GB            

diffusion_models/minimax_h3_ref2va_prune(…): downloading bytes:           |  0.00B            

[*] minimax_h3_video_vae_fp16.safetensors                      完成 4.9 GB
[*] minimax_h3_ref2va_pruned_int8_convrot.safetensors          完成 19.5 GB
[*] 全部文件就绪
[*] fl2va  就绪：minimax_h3_fl2va_pruned_int8_convrot.safetensors
[*] ref2va 就绪：minimax_h3_ref2va_pruned_int8_convrot.safetensors
/content/ComfyUI/models/diffusion_models:
total 40G
-rw-r--r-- 2 root root 20G Aug  8 14:30 minimax_h3_fl2va_pruned_int8_convrot.safetensors
-rw-r--r-- 2 root root 20G Aug  8 14:32 minimax_h3_ref2va_pruned_int8_convrot.safetensors
-rw-r--r-- 1 root root   0 Aug  8 14:28 put_diffusion_model_files_here

/content/ComfyUI/models/text_encoders:
total 26G
-rw-r--r-- 1 root root   0 Aug  8 14:28 put_text_encoder_files_here
-rw-r--r-- 2 root root 26G Aug  8 14:31 qwen3vl_32b_minimax_h3_int8_convrot.safetensors

/content/ComfyUI/models/vae:
total 5.5G
-rw-r--r-- 2 root root 578M Aug  8 14:31 minimax_h3_audio_vae_fp32.safetensors
-rw-r--r-- 2 root root 4.9G Aug  8 14:31 minimax_h3_video_vae_fp16.safetensors
-rw-r--r

In [6]:
# ==========================================================
# Cell 6: 把工作流改成“开箱即跑”并装进 ComfyUI
#   1) 三个 loader 的文件名 -> Cell 5 实际下到的文件
#   2) 显存不到 40G 时自动打开分段清显存
#   3) 开了 Turbo LoRA 就插一个 LoraLoaderModelOnly 到 UNETLoader 与导演台之间
#   4) 写到 user/default/workflows/，界面左侧工作流列表直接能点开
# ==========================================================
COMFY = CFG["comfy_dir"]
wf = wf_load()
changes = []

fam = CFG.get("model_family", "fl2va")
dit_files = CFG.get("dit_files") or {}
dit_base = os.path.basename(dit_files.get(fam) or CFG["dit_file"])   # 按当前任务选底模
te_base = os.path.basename(CFG["te_file"])
vae = CFG.get("vae_files", {})

for node, idx, sub, cur in wf_loaders(wf):
    new = None
    if node["type"] == "UNETLoader":
        new = dit_base
    elif node["type"] == "CLIPLoader":
        new = te_base
        wv = node["widgets_values"]
        if len(wv) > 1 and wv[1] != "minimax":
            wv[1] = "minimax"                       # type 必须是 minimax
            changes.append("CLIPLoader.type -> minimax")
    elif node["type"] == "VAELoader":
        title = (node.get("title") or "") + " " + cur
        new = vae.get("audio") if re.search(r"audio|音频", title, re.I) else vae.get("video")
    if new and new != cur:
        node["widgets_values"][idx] = new
        changes.append("%s: %s -> %s" % (node["type"], cur, new))

d = director(wf)
if d:
    tl, ti = timeline_get(d)

    # 小显存：分段之间清显存，不然第二段必 OOM
    if CFG.get("vram_gb", 0) < 38:
        j = widget_after(d, "性能", bool)
        if j >= 0 and d["widgets_values"][j] is not True:
            d["widgets_values"][j] = True
            changes.append("clear_vram_between_segments -> True")

    # Turbo LoRA：4 步
    if CFG.get("lora_ready"):
        j = widget_after(d, "高级采样", int)
        if j >= 0:
            d["widgets_values"][j] = 4
            changes.append("steps -> 4 (Turbo LoRA)")

    if tl:
        miss = [a for a in timeline_assets(tl)
                if not os.path.exists(os.path.join(COMFY, "input", a))]
        if miss:
            changes.append("还缺 %d 个首尾帧素材，跑 Cell 7 上传" % len(miss))

# ---------- 可选：插入 Turbo LoRA 节点 ----------
if CFG.get("lora_ready") and d and not wf_find(wf, "LoraLoaderModelOnly"):
    slot = next((i for i, inp in enumerate(d.get("inputs", []))
                 if inp.get("name") == "model"), None)
    link_id = d["inputs"][slot].get("link") if slot is not None else None
    old = next((l for l in wf["links"] if l[0] == link_id), None)
    if old:
        nid = max(n["id"] for n in wf["nodes"]) + 1
        lid = max([l[0] for l in wf["links"]] + [wf.get("last_link_id", 0)]) + 1
        wf["nodes"].append({
            "id": nid, "type": "LoraLoaderModelOnly",
            "title": "Turbo 4步 LoRA",
            "pos": [d["pos"][0], d["pos"][1] - 140], "size": [340, 82],
            "flags": {}, "order": 5, "mode": 0,
            "inputs": [{"name": "model", "type": "MODEL", "link": old[0]}],
            "outputs": [{"name": "MODEL", "type": "MODEL", "links": [lid], "slot_index": 0}],
            "properties": {"Node name for S&R": "LoraLoaderModelOnly"},
            "widgets_values": [CFG["lora_out"], 1.0],
        })
        old[3], old[4] = nid, 0                      # UNETLoader -> LoRA
        wf["links"].append([lid, nid, 0, d["id"], slot, "MODEL"])
        d["inputs"][slot]["link"] = lid              # LoRA -> 导演台
        wf["last_link_id"] = lid
        wf["last_node_id"] = max(wf.get("last_node_id", 0), nid)
        changes.append("插入 LoraLoaderModelOnly -> " + CFG["lora_out"])

# ---------- 加速模块：UNETLoader -> Sage 补丁 x2 -> 导演台 ----------
#   对应 AIMixer/ComfyUI_MiniMaxH3_Director 的
#   example_workflows/minimax_h3_director_加速版.json（作者 2026-08-07 新增）
ACCEL_CHAIN = [
    ("PathchSageAttentionKJ", "Patch Sage Attention KJ",
     ["auto", False], [270, 82], "MODEL"),
    ("MiniMaxH3MemoryEfficientSageAttentionPatch",
     "MiniMax H3 Mem Eff Sage Attention Patch", [], [330, 26], "model"),
]


def model_slot(node):
    return next((i for i, inp in enumerate(node.get("inputs", []))
                 if inp.get("name") == "model"), None)


def insert_accel(wf, d):
    """把补丁节点逐个串到上游与导演台之间（挂了 Turbo LoRA 就接在 LoRA 后面）。"""
    out = []
    mode = 0 if CFG.get("sage_ok") else 4      # 4 = 旁路，界面里 Ctrl+B 可开关
    for i, (ntype, title, widgets, size, oname) in enumerate(ACCEL_CHAIN):
        if wf_find(wf, ntype):
            continue
        slot = model_slot(d)
        if slot is None:
            out.append("导演台没有 model 输入口，跳过加速模块")
            break
        cur = d["inputs"][slot].get("link")
        old = next((l for l in wf["links"] if l[0] == cur), None)
        if old is None:
            out.append("找不到导演台的 model 连线，跳过加速模块")
            break
        nid = max(n["id"] for n in wf["nodes"]) + 1
        lid = max([l[0] for l in wf["links"]] + [wf.get("last_link_id", 0)]) + 1
        wf["nodes"].append({
            "id": nid, "type": ntype, "title": title,
            "pos": [d["pos"][0] + 60 * i, d["pos"][1] - 240 + 80 * i],
            "size": size, "flags": {}, "order": 5, "mode": mode,
            "inputs": [{"name": "model", "type": "MODEL", "link": old[0]}],
            "outputs": [{"name": oname, "type": "MODEL",
                         "links": [lid], "slot_index": 0}],
            "properties": {"Node name for S&R": ntype},
            "widgets_values": list(widgets),
        })
        old[3], old[4] = nid, 0                # 上游 -> 新节点
        wf["links"].append([lid, nid, 0, d["id"], slot, "MODEL"])
        d["inputs"][slot]["link"] = lid        # 新节点 -> 导演台
        wf["last_link_id"] = lid
        wf["last_node_id"] = max(wf.get("last_node_id", 0), nid)
        out.append("插入 %s%s" % (ntype, "" if mode == 0 else "（旁路）"))
    return out


if d and CFG.get("accel_sage") and CFG["sage_mode"] != "skip":
    changes += insert_accel(wf, d)
    if CFG.get("accel_steps"):
        j = widget_after(d, "高级采样", int)
        if j >= 0 and d["widgets_values"][j] != CFG["accel_steps"]:
            d["widgets_values"][j] = CFG["accel_steps"]
            changes.append("steps -> %d（加速版）" % CFG["accel_steps"])

wf_save(wf)

# ---------- 装进 ComfyUI（侧栏直接能打开）----------
wf_dir = os.path.join(COMFY, "user", "default", "workflows")
os.makedirs(wf_dir, exist_ok=True)
installed = os.path.join(wf_dir, CFG["workflow_name"] + ".json")
shutil.copy2(CFG["workflow_json"], installed)
CFG["workflow_installed"] = installed

# ---------- 可选：成片存 Drive ----------
if CFG["save_outputs_to_drive"] and os.path.ismount("/content/drive"):
    out_dir = os.path.join(CFG["drive_dir"], "output")
    os.makedirs(out_dir, exist_ok=True)
    local_out = os.path.join(COMFY, "output")
    if not os.path.islink(local_out):
        shutil.rmtree(local_out, ignore_errors=True)
        os.symlink(out_dir, local_out)
    log("输出目录已指向 Drive：" + out_dir)

print(BAR)
if changes:
    log("工作流已改写：")
    for c in changes:
        log("   " + c)
else:
    log("工作流无需改写")
log("已安装到 " + installed)
if dit_files:
    log("本地底模：")
    for f in sorted(dit_files):
        log("   %-7s %s%s" % (f, dit_files[f], "   <- 当前" if f == fam else ""))
    if len(dit_files) > 1:
        log("   换 r2v/v2v/rv2v：导演台改 task_type 后，把 UNETLoader 选成 ref2va 那个文件"
            "（或重跑 Cell 2 -> 6 自动回写）")
save_cfg()
print(BAR)


[*] 工作流已改写：
[*]    CLIPLoader: qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors -> qwen3vl_32b_minimax_h3_int8_convrot.safetensors
[*]    还缺 4 个首尾帧素材，跑 Cell 7 上传
[*]    插入 PathchSageAttentionKJ
[*]    插入 MiniMaxH3MemoryEfficientSageAttentionPatch
[*] 已安装到 /content/ComfyUI/user/default/workflows/MiniMaxH3_导演台全能工作流.json
[*] 本地底模：
[*]    fl2va   minimax_h3_fl2va_pruned_int8_convrot.safetensors   <- 当前
[*]    ref2va  minimax_h3_ref2va_pruned_int8_convrot.safetensors
[*]    换 r2v/v2v/rv2v：导演台改 task_type 后，把 UNETLoader 选成 ref2va 那个文件（或重跑 Cell 2 -> 6 自动回写）


In [7]:
# ==========================================================
# Cell 7: 首尾帧素材上传
#   工作流里的图片名是作者本地的哈希名，Colab 上当然不存在，
#   不处理的话一点 Run 就报 "invalid image file"。
#   现在：你按时间线顺序上传图片 -> 自动落到 input/ 并回写文件名与真实尺寸。
#   不想现在传：把 SKIP 改成 True，之后在 ComfyUI 界面里重新选图也行。
# ==========================================================
SKIP = False

COMFY = CFG["comfy_dir"]
IN_DIR = os.path.join(COMFY, "input")
os.makedirs(IN_DIR, exist_ok=True)

wf = wf_load()
d = director(wf)
tl, ti = timeline_get(d) if d else (None, -1)
needed = timeline_assets(tl) if tl else []
missing = [a for a in needed if not os.path.exists(os.path.join(IN_DIR, a))]

print(BAR)
log("时间线需要 %d 个素材，缺 %d 个" % (len(needed), len(missing)))
for i, a in enumerate(needed, 1):
    log("  %d. %s  %s" % (i, a, "✓" if a not in missing else "✗ 缺失"))

if SKIP or not missing:
    log("无需上传")
elif not IN_COLAB:
    log("非 Colab 环境，请自行把文件放到 " + IN_DIR, "!")
else:
    from google.colab import files
    print("\n按时间线顺序选图（可一次多选，按文件名排序对应上面的 1..%d）：" % len(needed))
    up = files.upload()
    names = sorted(up.keys())
    if not names:
        log("没有上传任何文件", "!")
    else:
        if len(names) != len(needed):
            log("上传 %d 个 / 需要 %d 个，按顺序尽量对应" % (len(names), len(needed)), "!")
        mapping = {}
        for old, new in zip(needed, names):
            dst = os.path.join(IN_DIR, os.path.basename(new))
            shutil.move(new, dst)
            mapping[old] = os.path.basename(new)
            log("%s  ->  %s" % (new, old))
        for n in names[len(needed):]:
            shutil.move(n, os.path.join(IN_DIR, os.path.basename(n)))

        # 把时间线里的旧文件名换成新名，并把宽高改成真实尺寸
        try:
            from PIL import Image
            def wh(fn):
                p = os.path.join(IN_DIR, fn)
                with Image.open(p) as im:
                    return im.size
        except Exception:
            def wh(fn):
                return None

        def fix(obj):
            if isinstance(obj, dict):
                for k in ("imageFile", "videoFile", "audioFile", "fileName"):
                    if obj.get(k) in mapping:
                        obj[k] = mapping[obj[k]]
                        s = wh(obj[k]) if k == "imageFile" else None
                        if s:
                            obj["width"], obj["height"] = s
                for v in obj.values():
                    fix(v)
            elif isinstance(obj, list):
                for v in obj:
                    fix(v)

        fix(tl)
        timeline_set(d, tl, ti)
        wf_save(wf)
        shutil.copy2(CFG["workflow_json"], CFG["workflow_installed"])
        log("时间线已回写新文件名与尺寸")

log("input/ 目录：")
sh("ls -lh %s | tail -n 20" % IN_DIR, check=False)
print(BAR)


[*] 时间线需要 4 个素材，缺 4 个
[*]   1. d47f7c7c243b746a437dd33ac21ed163192540329ac0784ed7e6bd83b440065a.png  ✗ 缺失
[*]   2. a1f4b9c98491d696b06b022b2588bf82b966a69356edc0fde952ce335820f26c.png  ✗ 缺失
[*]   3. aad0afcc1907dd5d8ce4f0c4e14f6511bde9cab23dde1da0311827d4ae4170d0.png  ✗ 缺失
[*]   4. 837b4f3722830bcbeabc138facc53c0e8fa90308543d98c2dde7e797b4a17cd4.png  ✗ 缺失

按时间线顺序选图（可一次多选，按文件名排序对应上面的 1..4）：


[!] 没有上传任何文件
[*] input/ 目录：
total 12K
-rw-r--r-- 1 root root 8.4K Aug  8 14:28 example.png


In [8]:
# ==========================================================
# Cell 8: 启动 ComfyUI + 外网通道 + 工作流可运行性体检
#   重启界面只需重跑 Cell 1 + 本格
# ==========================================================
import shlex, threading, urllib.request

CFG = load_cfg()
COMFY, PORT = CFG["comfy_dir"], CFG["local_port"]
assert os.path.isdir(COMFY), "先跑 Cell 3"

# ---------- 1. 显存策略 ----------
#   ComfyUI 只有 --gpu-only / --highvram / --lowvram / --novram / --cpu，
#   普通模式就是一个都不传，没有 --normalvram 这个参数。
vram = CFG.get("vram_gb", 0)
policy = CFG["vram_policy"]
if policy == "auto":
    policy = "normal" if vram >= 38 else ("lowvram" if vram >= 20 else "novram")
flags = "--cache-none --disable-smart-memory"
if policy != "normal":
    flags += " --" + policy
flags = (flags + " " + CFG["extra_args"]).strip()


def supported_flags():
    out = sh("%s main.py --help" % sys.executable, cwd=COMFY, check=False, quiet=True)
    return set(re.findall(r"--[A-Za-z0-9][A-Za-z0-9_.-]*", out))


def sanitize(extra, known):
    if not known:
        return extra, []
    toks, keep, dropped, i = shlex.split(extra), [], [], 0
    while i < len(toks):
        t = toks[i]
        if t.startswith("--"):
            good = t.split("=", 1)[0] in known
            (keep if good else dropped).append(t)
            i += 1
            while i < len(toks) and not toks[i].startswith("-"):
                if good:
                    keep.append(toks[i])
                i += 1
        else:
            keep.append(t)
            i += 1
    return " ".join(keep), dropped


flags, dropped = sanitize(flags, supported_flags())
if dropped:
    log("本版 ComfyUI 不认识 %s，已剔除" % " ".join(dropped), "!")
log("显存策略: %s | 参数: %s" % (policy, flags or "(无)"))

# ---------- 2. 启动 ----------
for f in (ROOT + "/comfy.log", ROOT + "/frpc.log", ROOT + "/cf.log"):
    if os.path.exists(f):
        os.remove(f)
sh("pkill -f frpc || true; pkill -f cloudflared || true; pkill -f 'ComfyUI/main.py' || true",
   check=False, quiet=True)

launch = ("python main.py --listen 127.0.0.1 --port %d --enable-cors-header '*' "
          "--preview-method auto --disable-auto-launch %s" % (PORT, flags))
log("launch: " + launch)
subprocess.Popen(launch + " > %s/comfy.log 2>&1" % ROOT, shell=True, cwd=COMFY)

ready = False
for i in range(300):
    time.sleep(2)
    txt = open(ROOT + "/comfy.log", errors="ignore").read() if os.path.exists(ROOT + "/comfy.log") else ""
    if "To see the GUI go to" in txt:
        ready = True
        break
    if "main.py: error:" in txt or ("Traceback" in txt and i > 15):
        log("启动报错，日志尾部：\n" + txt[-3000:], "!")
        break
log("ComfyUI 已就绪" if ready else "ComfyUI 未就绪", "*" if ready else "!")


# ---------- 3. 体检：节点在不在、文件名对不对、素材齐不齐 ----------
def api(path):
    with urllib.request.urlopen("http://127.0.0.1:%d%s" % (PORT, path), timeout=60) as r:
        return json.load(r)


problems = []
if ready:
    wf = wf_load()
    bypassed = {n.get("type") for n in wf["nodes"] if n.get("mode") == 4}
    for t in wf_types(wf):
        if t in ("Note", "MarkdownNote", "Reroute") or t in bypassed:
            continue          # 旁路(Ctrl+B)的节点不参与体检
        try:
            info = api("/object_info/" + t)
        except Exception:
            info = {}
        if not info.get(t):
            problems.append("缺节点 %s（重跑 Cell 4，或用 Manager 搜一下）" % t)
            continue
        req = (info[t].get("input", {}).get("required") or {})
        for node, idx, sub, fn in wf_loaders(wf):
            if node["type"] != t:
                continue
            field = list(req)[idx] if len(req) > idx else None
            opts = req.get(field, [None])[0] if field else None
            if isinstance(opts, list) and fn not in opts:
                problems.append("%s 选不到 %s（可选：%s）"
                                % (t, fn, ", ".join(map(str, opts[:4])) or "空"))
    d = director(wf)
    tl, _ = timeline_get(d) if d else (None, -1)
    for a in (timeline_assets(tl) if tl else []):
        if not os.path.exists(os.path.join(COMFY, "input", a)):
            problems.append("缺素材 input/%s（跑 Cell 7 上传）" % a)

# ---------- 4. 外网通道 ----------
#   frp = TCP 直连你自己的 frps，不经过任何公共边缘节点，延迟最低
url = "http://127.0.0.1:%d" % PORT
mode = CFG["tunnel"] if ready else "none"


def start_frp():
    FRP, v = CFG["frp_dir"], CFG["frp_ver"]
    if not os.path.exists(FRP + "/frpc"):
        sh("wget -qO- https://github.com/fatedier/frp/releases/download/v%s/"
           "frp_%s_linux_amd64.tar.gz | tar -xz -C %s" % (v, v, ROOT), check=False)
    if not os.path.exists(FRP + "/frpc"):
        log("frpc 没下下来（GitHub 出口不通）", "!")
        return None
    sh("chmod +x %s/frpc" % FRP, quiet=True)

    conf = ['serverAddr = "%s"' % CFG["frp_host"],
            "serverPort = %d" % CFG["frp_port"],
            "loginFailExit = false",
            "transport.tcpMux = true",
            "transport.poolCount = 5",
            "transport.heartbeatInterval = 15",
            "transport.heartbeatTimeout = 60",
            'log.to = "%s/frpc.log"' % ROOT,
            'log.level = "info"']
    if CFG["frp_token"]:
        conf.insert(3, 'auth.token = "%s"' % CFG["frp_token"])
    conf += ["", "[[proxies]]",
             'name = "comfyui_colab_%d"' % CFG["remote_port"],
             'type = "tcp"',
             'localIP = "127.0.0.1"',
             "localPort = %d" % PORT,
             "remotePort = %d" % CFG["remote_port"]]
    open(FRP + "/frpc.toml", "w").write("\n".join(conf) + "\n")
    log("frpc %s -> %s:%d，远端端口 %d"
        % (v, CFG["frp_host"], CFG["frp_port"], CFG["remote_port"]))

    subprocess.Popen("%s/frpc -c %s/frpc.toml >> %s/frpc.log 2>&1"
                     % (FRP, FRP, ROOT), shell=True)

    #   frps 侧常见拒绝原因，直接翻译成人话，不用去翻日志
    BAD = [("port not allowed",
            "frps 的 allowPorts 没放行 %d，在 frps.toml 里放行或把 remote_port 改成已放行的端口"
            % CFG["remote_port"]),
           ("port already used",
            "远端 %d 被占（上一个会话的 frpc 还挂着），等 30 秒重试或换 remote_port"
            % CFG["remote_port"]),
           ("token in login doesn't match", "frp_token 和 frps 对不上"),
           ("authorization failed", "frp_token 和 frps 对不上"),
           ("login to server failed",
            "连不上 %s:%d（frps 没跑 / 端口没开 / 域名解析不对）"
            % (CFG["frp_host"], CFG["frp_port"]))]
    txt = ""
    for _ in range(24):
        time.sleep(1.5)
        if os.path.exists(ROOT + "/frpc.log"):
            txt = open(ROOT + "/frpc.log", errors="ignore").read()
        if "start proxy success" in txt:
            return "http://%s:%d" % (CFG["frp_host"], CFG["remote_port"])
        for key, why in BAD:
            if key in txt:
                log("frp 失败：" + why, "!")
                return None
    log("frp 等了 36 秒没出 start proxy success，日志尾部：\n" + txt[-800:], "!")
    return None


def start_colab_proxy():
    if not IN_COLAB:
        return None
    from google.colab import output
    output.serve_kernel_port_as_window(PORT)
    return "Colab 端口代理新窗口（弹窗被拦就允许一下）"


def start_cloudflared():
    cf = ROOT + "/cloudflared"
    if not os.path.exists(cf):
        sh("wget -q -O %s https://github.com/cloudflare/cloudflared/releases/"
           "latest/download/cloudflared-linux-amd64" % cf, check=False)
        sh("chmod +x " + cf, quiet=True)
    subprocess.Popen("%s tunnel --no-autoupdate --url http://127.0.0.1:%d "
                     "--logfile %s/cf.log > /dev/null 2>&1" % (cf, PORT, ROOT), shell=True)
    for _ in range(40):
        time.sleep(2)
        txt = open(ROOT + "/cf.log", errors="ignore").read() if os.path.exists(ROOT + "/cf.log") else ""
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", txt)
        if m:
            return m.group(0)
    return None


if mode == "frp":
    got = start_frp()
    if not got and CFG.get("tunnel_fallback"):
        log("frp 没起来，先用 Colab 端口代理兔子底，修好 frps 后重跑本格即可", "!")
        got = start_colab_proxy()
    url = got or url
elif mode == "colab":
    url = start_colab_proxy() or url
elif mode == "cloudflared":
    got = start_cloudflared()
    if not got:
        log("cloudflared 没拿到域名", "!")
        got = start_colab_proxy() if CFG.get("tunnel_fallback") else None
    url = got or url

# ---------- 5. keep-alive，Colab 空闲 90 分钟会断 ----------
def _keep():
    while True:
        time.sleep(300)
        print("[keep-alive]", flush=True)


threading.Thread(target=_keep, daemon=True).start()

print("\n" + BAR)
print("ComfyUI  : %s" % ("READY" if ready else "NOT READY"))
print("访问地址 : %s" % url)
print("工作流   : 左侧侧栏 Workflows -> %s" % CFG["workflow_name"])
print("底模     : %s" % os.path.basename(CFG.get("dit_file", "")))
_alt = [v for k, v in (CFG.get("dit_files") or {}).items()
        if v != os.path.basename(CFG.get("dit_file", ""))]
if _alt:
    print("备用底模 : %s（换 r2v/v2v/rv2v 时在 UNETLoader 里选它）" % ", ".join(_alt))
print("文本编码 : %s" % os.path.basename(CFG.get("te_file", "")))
if problems:
    print("-" * 64)
    print("体检发现 %d 个问题：" % len(problems))
    for p in problems:
        print("  ✗ " + p)
else:
    print("体检     : 节点 / 模型文件 / 素材 全部对得上，可以直接 Run")
print(BAR + "\n")
if not ready:
    print(open(ROOT + "/comfy.log", errors="ignore").read()[-3000:])

subprocess.run("tail -f %s/comfy.log" % ROOT, shell=True)


[*] 显存策略: normal | 参数: --cache-none --disable-smart-memory
[*] launch: python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header '*' --preview-method auto --disable-auto-launch --cache-none --disable-smart-memory
[*] ComfyUI 已就绪
[*] frpc 0.56.0 -> usoren.usdream.dpdns.org:7000，远端端口 8091

ComfyUI  : READY
访问地址 : http://usoren.usdream.dpdns.org:8091
工作流   : 左侧侧栏 Workflows -> MiniMaxH3_导演台全能工作流
底模     : minimax_h3_fl2va_pruned_int8_convrot.safetensors
备用底模 : minimax_h3_ref2va_pruned_int8_convrot.safetensors（换 r2v/v2v/rv2v 时在 UNETLoader 里选它）
文本编码 : qwen3vl_32b_minimax_h3_int8_convrot.safetensors
----------------------------------------------------------------
体检发现 4 个问题：
  ✗ 缺素材 input/d47f7c7c243b746a437dd33ac21ed163192540329ac0784ed7e6bd83b440065a.png（跑 Cell 7 上传）
  ✗ 缺素材 input/a1f4b9c98491d696b06b022b2588bf82b966a69356edc0fde952ce335820f26c.png（跑 Cell 7 上传）
  ✗ 缺素材 input/aad0afcc1907dd5d8ce4f0c4e14f6511bde9cab23dde1da0311827d4ae4170d0.png（跑 Cell 7 上传）
  ✗ 缺素材 input/837b4f3722830

KeyboardInterrupt: 

In [9]:
import torch, sys
print(torch.__version__, torch.version.cuda, sys.version)
print(torch.cuda.get_device_capability())   # A100 应为 (8, 0)

2.11.0+cu128 12.8 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
(8, 0)


In [10]:
!nvcc --version
!echo $CUDA_HOME; ls /usr/local/cuda/bin/nvcc

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

/usr/local/cuda/bin/nvcc


In [12]:
%cd /content
!rm -rf SageAttention
!git clone --depth 1 https://github.com/thu-ml/SageAttention.git
%cd /content/SageAttention
!CUDA_HOME=/usr/local/cuda \
 TORCH_CUDA_ARCH_LIST="8.0" \
 EXT_PARALLEL=4 NVCC_APPEND_FLAGS="--threads 8" MAX_JOBS=4 \
 pip install . --no-build-isolation

/content
Cloning into 'SageAttention'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 160 (delta 23), reused 88 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 50.46 MiB | 19.68 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/SageAttention
Processing /content/SageAttention
  Preparing metadata (pyproject.toml) ... done
[keep-alive]
  Created wheel for sageattention: filename=sageattention-2.2.0-cp312-cp312-linux_x86_64.whl size=8701313 sha256=94b863aec24f58c5a2bb30ec34ec81c2ca6df6ceee6577169904838318ea169b
  Stored in directory: /tmp/pip-ephem-wheel-cache-s2mf31hy/wheels/d2/9b/69/f02fe64f8a5160279c4252690064b8924309138319725a814c
Successfully built sageattention
  Attempting uninstall: sageattention
    Found existing installation: sageattention 1.0.6
    Uninstalling sageattention-1.0.6:
      Successfully uninstalled sageattention-1.0.6


In [13]:
!python -c "from sageattention import sageattn_qk_int8_pv_fp16_cuda; import sageattention._qattn_sm80 as m; print('OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/SageAttention/sageattention/__init__.py", line 1, in <module>
    from .core import sageattn, sageattn_varlen
  File "/content/SageAttention/sageattention/core.py", line 47, in <module>
    from .quant import per_block_int8 as per_block_int8_cuda
  File "/content/SageAttention/sageattention/quant.py", line 20, in <module>
    from . import _fused
ImportError: cannot import name '_fused' from partially initialized module 'sageattention' (most likely due to a circular import) (/content/SageAttention/sageattention/__init__.py)


In [14]:
%cd /content
!python -c "from sageattention import sageattn_qk_int8_pv_fp16_cuda; import sageattention._qattn_sm80, sageattention, os; print('OK', os.path.dirname(sageattention.__file__))"

/content
OK /usr/local/lib/python3.12/dist-packages/sageattention


In [16]:
%%writefile /content/t.py
import torch
from sageattention import sageattn_qk_int8_pv_fp16_cuda as f
q = torch.randn(1, 8, 1024, 128, dtype=torch.float16, device="cuda")
o = f(q, q, q, tensor_layout="HND")
torch.cuda.synchronize()
print("kernel OK", o.shape, o.dtype)

Writing /content/t.py


In [17]:
!cd /content && python t.py

kernel OK torch.Size([1, 8, 1024, 128]) torch.float16


## 报错速查

| 现象 | 原因 | 处理 |
| --- | --- | --- |
| 节点变红框 `MiniMaxH3Director` | 导演台插件没装上或没重启 | 重跑 Cell 4，再重跑 Cell 8（新节点要重启服务才加载） |
| 下拉框里没有模型文件 | ComfyUI 只在启动时扫目录 | 界面右上角 Refresh，或重跑 Cell 8 |
| `[Errno 2] No such file or directory: models/vae/xxx.safetensors`，`ls` 里却看得到 | 把 HF 缓存的相对符号链接硬链过去了，悬空 | v4 已修（先 realpath）。手动清：`find /content/ComfyUI/models -xtype l -delete` 后重跑 Cell 5 |
| `invalid image file` / 首尾帧报错 | `input/` 里没有时间线引用的图 | 跑 Cell 7 上传，或在导演台面板里重新选图 |
| `CUDA out of memory` | 分辨率/帧数超了本档显存 | 导演台里把 megapixels 降到 0.3；每段 124 帧 → 85 帧；确认分段清显存已开 |
| 会话直接死掉 / 重连 | 系统内存爆了 | 运行时改成高 RAM；保持 `--cache-none`；先只跑一段验证 |
| `main.py: error: unrecognized arguments` | 自己加的 `extra_args` 本版不认识 | Cell 8 会自动剔除并提示；显存档位只有 `--gpu-only/--highvram/--lowvram/--novram/--cpu` |
| `CUDA error: no kernel image is available` | ComfyUI requirements 把 torch 换掉了 | 按 Cell 3 打印的旧版本号 `pip install -q torch==<旧版>`，然后重启运行时 |
| `Only a single TORCH_LIBRARY ... triton` | 在内核里 reload 了 torch | 不要在单元格里 `import torch`；已发生就重启运行时，只跑 Cell 1 + 8 |
| frp `port not allowed` | frps 的 `allowPorts` 不含 8091 | frps.toml 里放行，或把 `remote_port` 改成已放行的端口 |
| frp `port already used` | 上一个会话的 frpc 还挂在 frps 上 | 等 30 秒重跑 Cell 8，或换一个 `remote_port` |
| frp `login to server failed` | frps 没跑 / 7000 没开 / 域名解析不对 | 先在本地 `telnet frp_host 7000` 试一下 |
| frp `token ... doesn't match` | `frp_token` 和 frps 对不上 | Cell 1 填对 `frp_token` |
| 换了 r2v / v2v / rv2v 任务报形状错 | 这些任务要 `ref2va` 底模，你还挂着 fl2va | 两套底模默认都已在本地：把 UNETLoader 换成 `minimax_h3_ref2va_*` 那个文件即可，或重跑 Cell 2 → 6 自动回写 |
| 加速节点报 `sageattention ... required` 或内核错误 | PyPI 上的 sageattention 是 1.x，`MiniMaxH3MemoryEfficientSageAttentionPatch` 要 2.x 内核 | 选中该节点 `Ctrl+B` 旁路，只留 `Patch Sage Attention KJ`；或自行源码编译 SageAttention 2 |
| 开了加速后画面和之前不一样 | 加速走的是近似注意力（int8/fp8 量化） | 正常现象；要同 seed 严格比对就把加速模块旁路 |

---

## 这个工作流里的默认参数

| 项 | 值 | 说明 |
| --- | --- | --- |
| 任务 | `fl2v` 首尾帧生视频 | 对应 `fl2va` 底模 |
| 分辨率 | 576 × 736（3:4 竖版 0.4 MP） | A100 下很宽裕 |
| 分段 | 3 段 × 124 帧 @ 24fps | 共 372 帧，约 15.5 秒 |
| steps / sampler | 20 / `res_multistep` + `simple` | 不挂 LoRA 的正常档 |
| shift video / audio | 12 / 3 | 已验证组合，别改 |
| cfg | 1.0 | 等价 BasicGuider |

想用 **Turbo 4 步 LoRA**：Cell 1 把 `use_turbo_lora` 改 `True`。
注意它必须配 **非剪枝 34 GB 底模**（LoRA 补的是完整 `adaln_proj (96768, 2688)`，
剪枝版只有 `(96768, 8)`，会在 51 处报形状不匹配），总占盘会从 ≈55 GB 涨到 ≈68 GB，
Cell 5 会自动改下非剪枝版，Cell 6 会插入 LoRA 节点并把 steps 改成 4。

---

## 存盘与提速

- `use_drive = True`：把 Drive 挂上；`save_outputs_to_drive = True` 会把 `ComfyUI/output` 指向 Drive，会话断了成片还在。
- 模型本体不建议放 Drive：27 GB 文本编码器从 Drive 读一次比重下一次还慢。
- **加速模块**：`accel_sage = True`（默认）会让 Cell 4 装好 KJNodes + sageattention，Cell 6 自动把 `UNETLoader → PathchSageAttentionKJ → MiniMaxH3MemoryEfficientSageAttentionPatch → 导演台` 串起来，对应作者 2026-08-07 新增的 `minimax_h3_director_加速版.json`。
- sageattention 装不上时，两个节点仍会插入但处于**旁路**状态，选中按 `Ctrl+B` 即可开关，出片不受影响。
- 加速是近似计算（Q/K 量化到 int8、V 到 fp8），画面与不开时会有细微差别；要严格做同 seed 比对就把它旁路掉。
- 不想要就把 `accel_sage` 设成 `False`，或把 `sage_mode` 设成 `"skip"`。
- 导演台面板里的 `Ollama` 提示词扩写需要本地 Ollama 服务，Colab 上没有，不要点那个按钮。
